# Instalation


In [1]:
!pip install transformers==4.53 accelerate datasets evaluate sentencepiece safetensors peft bitsandbytes torch torchvision
!pip install vllm==0.11.0
# Removed this:
# ==0.5.3.post1


  Using cached transformers-4.53.0-py3-none-any.whl.metadata (39 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.53.0-py3-none-any.whl (10.8 MB)
Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.11.0 requires transformers>=4.55.2, but you have transformers 4.53.0 which is incompatible.
  Using cached transformers-4.57.1-py3-non

# Preparation

## Huggingface Login

In [2]:
from huggingface_hub import notebook_login

notebook_login()

## Drive Login

In [3]:
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/ColabNotebooks/medical-argumentation'

Mounted at /content/drive


# Code

## Imports

In [4]:
import argparse
import json
import os
from pathlib import Path

# import torch
from vllm import LLM, SamplingParams
import torch

# DEVICE_NUM = torch.cuda.device_count()
#os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

INFO 11-20 16:35:29 [__init__.py:216] Automatically detected platform cuda.


## Prompt

In [5]:
SYSPROMPT = "You are a helpful medical assistant, expert clinical argumentation and who knows Spanish.\n"
PROMPT = """
You are given a justification for a multiple-choice medical exam question. The justification is written in an option-oriented style, where the reasoning is structured around eliminating or supporting specific answer choices. Your task is to rewrite this justification in a natural, clinical reasoning format, as a clinician would explain their diagnostic thinking in real-life practice.

Instructions:
- Do not refer to options or answer numbers (e.g., avoid "Option 1 is incorrect").
- Present the reasoning as a flowing, cohesive diagnostic argument.
- Keep all relevant medical content and logic, but phrase it in a neutral, clinician-oriented way.
- Use appropriate clinical language, as would be found in case presentations, consults, or differential diagnosis discussions.
- Avoid exam-like phrasing such as “this makes option X incorrect.”

Some evidences are marked in the argumentation between '#' or '@' characters:
- '@' indicates an attack relation.
- '#' may indicate a support relation or an attack relation if indicated in parenthesis (i.e.: #evidence#(rechaza)#).

Example:

Options:
1. - Carcinoma indiferenciado nasofaríngeo.
2. - Adenocarcinoma de base de lengua.
3. - Carcinoma mucoepidermoide de hipofaringe.
4. - Carcinoma epidermoide de laringe.

Argumentation:
correcta 4: ante un paciente de edad media, varón, fumador, que presenta disfonía y una masa laterocervical asociada a disnea, debemos pensar en carcinoma epidermoide de laringe como primera sospecha diagnóstica. #fumador# #disfonía# #masa laterocervical pétrea y cierta dificultad respiratoria#      incorrecta 1: el carcinoma de nasofaringe no tendría que cursar con disfonía por su localización @disfonía@ y tampoco se ha relacionado con el tabaquismo @fumador@      incorrectas 2 y 3: estas localizaciones anatómicas tampoco deberían cursar con disfonía@disfonía@

Output Example:
En un paciente de mediana edad, varón, fumador, que consulta por disfonía y presenta una masa laterocervical pétrea acompañada de cierta dificultad respiratoria, la sospecha diagnóstica principal debe orientarse hacia un carcinoma epidermoide de laringe. La presencia de disfonía sugiere afectación directa de las cuerdas vocales o estructuras adyacentes de la laringe, lo cual no es habitual en neoplasias de localización nasofaríngea, lingual posterior o hipofaríngea. Además, el antecedente de tabaquismo refuerza aún más la posibilidad de una neoplasia laríngea, ya que es un factor de riesgo conocido para este tipo de tumor.


Actual Instance:

Options:
{option_text}

Argumentation:
{argumentation}

Output:
"""

## Inference Functions

In [6]:
def prepare_prompt(question: dict):
    """
    Generate the prompt for a given question.
    :param question: The question dictionary that contains the argumentation and options.
    :return: The formatted prompt as a list of messages.
    """

    argument = question["first_argument"]
    option_text = '\n'.join(f'{idx}. - {option}' for idx, option in enumerate(question["options"], start=1))
    return [
        {"role": "system", "content": SYSPROMPT},
        {"role": "user", "content": PROMPT.format(argumentation=argument, option_text=option_text)}
    ]


def separate_thinking_from_response(generated_text, think_start_token, think_end_token):
    """
    Separate the thinking process from the final response if there is any thinking process.

    :param generated_text: Output generated by the model.
    :param think_start_token: Token indicating the start of the thinking process.
    :param think_end_token: Token indicating the end of the thinking process.
    :return: A tuple (thinking, response). Thinking tokens are removed. If no thinking process, thinking is None. If thinking process is not finished, response is None.
    """
    thinking = None
    response = generated_text

    # If there is a thinking process, separate it
    if generated_text.startswith(think_start_token):
        generated_text = generated_text[len(think_start_token):]

        # Check if thinking process has ended
        if (think_end_token_offset := generated_text.find(think_end_token)) != -1:
            thinking = generated_text[:think_end_token_offset]
            response = generated_text[think_end_token_offset + len(think_end_token):]
        else:
            thinking = generated_text
            response = None

    return thinking, response


def parse_argumentation(input_file: Path, output_folder: Path, model: str, think_start_token: str, think_end_token: str):
    output_folder.mkdir(parents=True, exist_ok=True)

    # Load the llm
    llm = LLM(
        model=model,
        limit_mm_per_prompt={"image": 0},       # Disable multimodality

        tensor_parallel_size=1,                 # Ensure the use of 1 GPU
        max_model_len=704,                     # Reduce max model size for better memory allocation
        max_num_batched_tokens=704,
        gpu_memory_utilization=0.4,             # Optimize memory utilization
        cpu_offload_gb=3,                       # CPU Offloading
        max_num_seqs=1,                         # One instance per batch size (memory optimization)
        enforce_eager=True,                     # Memory optimization
        mm_processor_cache_gb=0,
    )

    sampling_params = SamplingParams(
        temperature=0.4,
        top_k=10,
        top_p=0.8,
        max_tokens=3000,
    )

    # Process the dataset
    data = json.loads(input_file.read_text(encoding='utf-8'))
    prompts = [prepare_prompt(question) for question in data]

    # Make inference
    outputs = llm.chat(
        prompts,
        sampling_params=sampling_params,
        add_generation_prompt=True,
    )

    # Process outputs
    for question, output in zip(data, outputs):
        generated_text = output.outputs[0].text
        thinking, response = separate_thinking_from_response(generated_text, think_start_token, think_end_token)
        question["neutralized_argumentation"] = response
        question["neutralization_thinking_process"] = thinking

    # Save the results
    (output_folder / f"{model.split('/')[-1]}--{input_file.name}").write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')

In [7]:
import gc
import torch

# Flush memory
gc.collect()
torch.cuda.empty_cache()

# Inference

In [8]:
parse_argumentation(
    input_file=Path(f'{BASE_PATH}/data/original/es_example.json'),
    output_folder=Path(f'{BASE_PATH}/data//neutralized'),
    model='unsloth/medgemma-27b-it-GGUF',
    think_start_token='<unused94>',
    think_end_token='<unused95>',
)


INFO 11-20 16:35:47 [utils.py:233] non-default args: {'max_model_len': 704, 'cpu_offload_gb': 3, 'gpu_memory_utilization': 0.4, 'max_num_batched_tokens': 704, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 0}, 'mm_processor_cache_gb': 0, 'model': 'unsloth/medgemma-27b-it-GGUF'}


ValidationError: 1 validation error for ModelConfig
  Value error, Invalid repository ID or local directory specified: 'unsloth/medgemma-27b-it-GGUF'.
Please verify the following requirements:
1. Provide a valid Hugging Face repository ID.
2. Specify a local directory that contains a recognized configuration file.
   - For Hugging Face models: ensure the presence of a 'config.json'.
   - For Mistral models: ensure the presence of a 'params.json'.
3. For GGUF: pass the local path of the GGUF checkpoint.
   Loading GGUF from a remote repo directly is not yet supported.
 [type=value_error, input_value=ArgsKwargs((), {'model': ...rocessor_plugin': None}), input_type=ArgsKwargs]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error